In [ ]:
from pathlib import Path
import re
import xarray as xr
import pdb
fb_dir = Path(
    "/g/data/gv90/ae8456/SWOT_SEAICE/DATA/"
    "020_010W/SWOT_FREEBOARD_020_010W"
)

l2_files = sorted(
    fb_dir.glob("SWOT_L2_LR_SSH_Unsmoothed_*_crop_freeboard.nc")
)


l3_files = sorted(
    fb_dir.glob("SWOT_L3_LR_SSH_Unsmoothed_*_crop_freeboard.nc")
)

print(f"Found {len(l2_files)} L2 Unsmoothed freeboards")


def get_cycle_pass(path):
    m = re.search(
        r"SWOT_L[23]_LR_SSH_Unsmoothed_(\d{3})_(\d{3})_",
        path.name
    )

    if m:
        return int(m.group(1)), int(m.group(2))

    return None, None

In [ ]:
l2_by_pass = {
    get_cycle_pass(f): f
    for f in l2_files
    if get_cycle_pass(f) != (None, None)
}

l3_by_pass = {
    get_cycle_pass(f): f
    for f in l3_files
    if get_cycle_pass(f) != (None, None)
}
matching_cycle_pass = sorted(
    set(l2_by_pass) & set(l3_by_pass)
)

print(f"Matching L2/L3 pairs: {len(matching_cycle_pass)}")

In [ ]:
matched_files = []

for cycle_pass in matching_cycle_pass:

    cycle, pass_num = cycle_pass

    matched_files.append({
        "cycle": cycle,
        "pass": pass_num,
        "l2": l2_by_pass[cycle_pass],
        "l3": l3_by_pass[cycle_pass],
    })

    print(
        f"Cycle {cycle:03d}, Pass {pass_num:03d}"
    )

In [ ]:
lon_range_180 = (-20.713, -24.672)
lat_range = (-72.812, -74.668)

lon_min, lon_max = sorted(lon_range_180)
lat_min, lat_max = sorted(lat_range)

print(lon_min, lon_max)
print(lat_min, lat_max)

In [ ]:
candidate_files = []

for item in matched_files[:1]:

    cycle = item["cycle"]
    pass_num = item["pass"]

    l2_file = item["l2"]
    l3_file = item["l3"]

    with xr.open_dataset(l3_file) as ds:
        #pdb.set_trace()
        lon = ds["longitude"]
        lat = ds["latitude"]
        fb = ds["freeboard_v2"]

        inside = (
            (lon >= lon_min) &
            (lon <= lon_max) &
            (lat >= lat_min) &
            (lat <= lat_max)
        )

        valid_inside = inside & fb.notnull()

        if bool(valid_inside.any()):

            candidate_files.append(item)

            n_pixels = int(valid_inside.sum())

            print(
                f"Cycle {cycle:03d}, "
                f"Pass {pass_num:03d}: "
                f"{n_pixels} valid L2 pixels in bbox"
            )

print(f"\n{len(candidate_files)} matched L2/L3 passes intersect bbox")

In [ ]:
pass_num